<a href="https://colab.research.google.com/github/Chanaporn042/GE338-Lab-2/blob/main/Lab2_landsat8_6606614672.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.7 MB/s eta 0:00:00


In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-chanaporn040')

In [ ]:
import ee
import geemap

# =======================
# 1. กำหนดพื้นที่
# =======================

gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")
phichit = gaul.filter(ee.Filter.eq('ADM1_NAME', 'Phichit'))

Map = geemap.Map(center=[16.4, 100.3], zoom=9)
Map.addLayer(phichit, {}, 'Phichit Boundary')


# =======================
# 2. ฟังก์ชัน mask เมฆสำหรับ Landsat 8 SR
# =======================
def maskL8sr(image):
    # QA_PIXEL bits: cloud shadow = 3, clouds = 5
    cloudShadowBitMask = 1 << 3
    cloudsBitMask = 1 << 5
    mask = (image.select('QA_PIXEL').bitwiseAnd(cloudShadowBitMask).eq(0)
            .And(image.select('QA_PIXEL').bitwiseAnd(cloudsBitMask).eq(0)))
    # เลือก band ที่ใช้ (Green, Red, NIR, SWIR1) และแปลงเป็น reflectance
    return (image.updateMask(mask)
            .select(['SR_B3', 'SR_B4', 'SR_B5', 'SR_B6'])
            .multiply(0.0000275).add(-0.2))


# =======================
# 3. โหลด Landsat 8
# =======================
def getCompositeL8(start, end):
    collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(phichit)
        .filterDate(start, end)
        .map(maskL8sr))

    return collection.median().clip(phichit)


l8_2020 = getCompositeL8('2020-01-01', '2020-04-30')
l8_2024 = getCompositeL8('2024-01-01', '2024-04-30')


# =======================
# 4. คำนวณ Index
# =======================
# NDVI = (NIR - Red)/(NIR + Red) => SR_B5 - SR_B4
ndvi_2020 = l8_2020.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
ndvi_2024 = l8_2024.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

# NDWI = (Green - NIR)/(Green + NIR) => SR_B3 - SR_B5
ndwi_2020 = l8_2020.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
ndwi_2024 = l8_2024.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')

# NDMI = (NIR - SWIR1)/(NIR + SWIR1) => SR_B5 - SR_B6
ndmi_2020 = l8_2020.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI')
ndmi_2024 = l8_2024.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI')


# =======================
# 5. Change detection
# =======================
ndvi_change = ndvi_2024.subtract(ndvi_2020)
ndmi_change = ndmi_2024.subtract(ndmi_2020)


# =======================
# 6. Visualization
# =======================
ndviVis = {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}
ndwiVis = {'min': -1, 'max': 1, 'palette': ['brown', 'white', 'blue']}
changeVis = {'min': -0.5, 'max': 0.5, 'palette': ['red', 'white', 'green']}

Map.addLayer(ndvi_2020, ndviVis, 'NDVI 2020')
Map.addLayer(ndvi_2024, ndviVis, 'NDVI 2024')
Map.addLayer(ndwi_2024, ndwiVis, 'NDWI 2024')
Map.addLayer(ndmi_2024, ndviVis, 'NDMI 2024')

Map.addLayer(ndvi_change, changeVis, 'NDVI Change')
Map.addLayer(ndmi_change, changeVis, 'NDMI Change')

Map

Map(center=[16.4, 100.3], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI…

In [ ]:
# 5. Export NDMI ไป Google Drive
# ======================================
task = ee.batch.Export.image.toDrive(
    image=ndvi_2024,
    description='Phichit_NDVI_Change',
    folder='Lab2',        # Folder บน Google Drive
    fileNamePrefix='ndvi_change_landsat8', # ชื่อไฟล์
    region=phichit.geometry(),
    scale=20,                   # 🔹 ใช้ 20m ลด memory
    crs='EPSG:4326',
    maxPixels=1e13
)

task.start()
print("Export started! ตรวจสอบสถานะได้ด้วย task.status()")

Export started! ตรวจสอบสถานะได้ด้วย task.status()
